<a href="https://colab.research.google.com/github/VertualDart/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VertualDart/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
Lane: Refresh / Content Opportunity Scoring (Lane 2).

I want to investigate which pages a content reviewer should look at first when review time is limited. I chose this lane because the starter dataset contains several useful page-level signals, such as impressions, position, CTR, word count, and update history. It also connects directly to a practical decision: instead of asking a reviewer to manually inspect thousands of pages, can we produce a useful priority order for review? The lane is provisional, so I can change it later if the data shows that another question is more useful. For now, the goal is to establish whether the starter data contains enough signal to justify building a ranking or scoring approach.

In [ ]:
chosen_lane = "Refresh / Content Opportunity Scoring"
lane_number = 2

print(f"Provisional lane: {chosen_lane}")
print(f"Lane number: {lane_number}")

Provisional lane: Refresh / Content Opportunity Scoring
Lane number: 2


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision: Which pages should be reviewed for a possible refresh first?

Who acts on the output: A content strategist or SEO reviewer.

Action: The reviewer uses the ranked list to decide which pages to inspect first. After reviewing a page, they may refresh it, consolidate it with another page, make a smaller change, or leave it as it is.

Cost of a wrong recommendation: A false positive can waste limited reviewer time on a page that does not need attention. A false negative can cause a page that may need attention to be missed. The second cost can also mean that an opportunity to investigate a declining page is delayed. Because these costs are different, I care more about getting useful pages near the top of the review list than about simply maximizing overall classification accuracy.

This makes ranking quality, such as precision@K, a useful metric to consider later. The eventual system should support a human reviewer rather than automatically deciding what happens to a page.

In [ ]:
decision_frame = {
    "decision": "Which pages should be reviewed for a possible refresh first?",
    "actor": "Content strategist / SEO reviewer",
    "action": "Inspect the page and decide whether to refresh, consolidate, change, or leave it",
    "false_positive_cost": "Wasted reviewer time on a page that did not need attention",
    "false_negative_cost": "A page that may need attention is missed or reviewed later",
    "candidate_success_metric": "precision@K for the ranked review queue",
}

for key, value in decision_frame.items():
    print(f"{key}: {value}")

decision: Which pages should be reviewed for a possible refresh first?
actor: Content strategist / SEO reviewer
action: Inspect the page and decide whether to refresh, consolidate, change, or leave it
false_positive_cost: Wasted reviewer time on a page that did not need attention
false_negative_cost: A page that may need attention is missed or reviewed later
candidate_success_metric: precision@K for the ranked review queue


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/VertualDart/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())

Dataset shape: (30000, 44)
Unique clients: 32


In [ ]:
# Number 1: pages showing a downward trend while still having
# at least 100 impressions in the 90-day window.

declining_with_demand = (
    (df["trend_direction"] == "down")
    & (df["impressions_90d"] >= 100)
)

n_declining = int(declining_with_demand.sum())
pct_declining = 100 * n_declining / len(df)

print(
    f"Pages flagged declining_with_demand: "
    f"{n_declining:,} of {len(df):,} ({pct_declining:.1f}%)"
)

Pages flagged declining_with_demand: 13,152 of 30,000 (43.8%)


In [ ]:
# Number 2: how much of the dataset's 90-day impression volume
# is associated with those declining pages.

total_impressions = int(df["impressions_90d"].sum())
declining_impressions = int(
    df.loc[declining_with_demand, "impressions_90d"].sum()
)

pct_impressions = 100 * declining_impressions / total_impressions

print(f"Total 90-day impressions: {total_impressions:,}")
print(
    f"Impressions on declining-with-demand pages: "
    f"{declining_impressions:,} ({pct_impressions:.1f}%)"
)

Total 90-day impressions: 156,010,989
Impressions on declining-with-demand pages: 79,887,612 (51.2%)


In [ ]:
# Number 3: pages with at least 500 impressions and a downward trend.

high_impression_declining = (
    (df["impressions_90d"] >= 500)
    & (df["trend_direction"] == "down")
)

n_high_impression_declining = int(high_impression_declining.sum())

print(
    "Pages with >=500 impressions and a downward trend:",
    f"{n_high_impression_declining:,}"
)

Pages with >=500 impressions and a downward trend: 9,961


### What these numbers suggest

The starter data shows that 13,152 of 30,000 pages (43.8%) are currently in the `declining_with_demand` group used for this initial check. Those pages account for about 79.9 million of the dataset's 156.0 million 90-day impressions (51.2%). There are also 9,961 pages with at least 500 impressions and a downward trend.

These measurements make the refresh-scoring question worth investigating because the potential review pool is large and includes pages with substantial measured impression volume. The numbers alone do not tell me which pages should actually be refreshed, so a ranking approach still needs to be tested against an appropriate observed outcome.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I can say now

I can say that the starter dataset contains a substantial number of pages with a measured downward trend and meaningful 90-day impression volume. I can also measure associations between the available page-level signals and these observed labels.

### What I can investigate later

I can investigate whether other page-level signals can help rank pages that are more likely to be worth reviewing. A stronger version of the project would use a future outcome window from the daily warehouse data, if the available history supports it.

### What I cannot claim

I cannot claim that a page will recover because it is refreshed. This dataset is observational, so it cannot establish that causal relationship.

I also cannot claim that the work predicts Google's ranking algorithm. The project is about decision-support for prioritizing content review.

Finally, `trend_direction` and `trend_pct` must not be used as model features later. `trend_direction` is derived from `trend_pct`, and the framing rules warn against using a derived label or its source as a feature. For this week, I am only using the current `trend_direction` to understand the starter data and establish whether the lane is worth pursuing.

In [ ]:
claims = {
    "observed": "Describe measured patterns in the starter dataset",
    "directional": "Use patterns to decide whether the lane is worth further investigation",
    "decision_support": "Aim to prioritize human content review",
    "not_causal": "Do not claim that refreshing a page causes recovery",
    "not_google_prediction": "Do not claim to predict Google's ranking system",
}

for claim_type, description in claims.items():
    print(f"{claim_type}: {description}")

observed: Describe measured patterns in the starter dataset
directional: Use patterns to decide whether the lane is worth further investigation
decision_support: Aim to prioritize human content review
not_causal: Do not claim that refreshing a page causes recovery
not_google_prediction: Do not claim to predict Google's ranking system


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.